# EECE 5644 Project – Data Exploration (LISA Traffic Light Dataset)
**Team:** Jingqi Mei & Andrew Chen

Every section ends with a **Decision** cell: EDA here is meant to fix concrete choices for the pipeline
(patch size, minimum box size, frame subsampling, class handling, CV fold design), not just to make plots.

Only the annotation CSVs are needed for Sections 1–5; Sections 6–7 also need the frames.

In [ ]:
import os, glob, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import cv2


In [4]:

# >>> Adjust to your download location 
ROOT = "./data"
ANN  = os.path.join(ROOT, "Annotations", "Annotations")
CLASSES = ["go", "goLeft", "stop", "stopLeft", "warning", "warningLeft"]
COLOR_OF = {"go": "green", "goLeft": "green", "goForward": "green",
            "stop": "red", "stopLeft": "red", "warning": "yellow", "warningLeft": "yellow"}
pd.set_option("display.width", 200)

## 1. Load all box annotations

In [5]:
rows = []
for f in sorted(glob.glob(f"{ANN}/**/frameAnnotationsBOX.csv", recursive=True)):
    df = pd.read_csv(f, sep=";")
    clip = os.path.relpath(os.path.dirname(f), ANN)          # e.g. dayTrain/dayClip1 or daySequence1
    df["clip"]   = clip.replace(os.sep, "/")
    df["split"]  = "train" if "Train" in clip else "test"     # 18 training clips vs 4 held-out sequences
    df["isNight"] = clip.lower().startswith("night")
    rows.append(df)
ann = pd.concat(rows, ignore_index=True)
ann = ann.rename(columns={"Annotation tag": "tag",
                          "Upper left corner X": "x1", "Upper left corner Y": "y1",
                          "Lower right corner X": "x2", "Lower right corner Y": "y2"})
ann["frame"] = ann["Filename"].map(os.path.basename)
print(ann.shape); print(ann.columns.tolist())
ann.head()

(109475, 14)
['Filename', 'tag', 'x1', 'y1', 'x2', 'y2', 'Origin file', 'Origin frame number', 'Origin track', 'Origin track frame number', 'clip', 'split', 'isNight', 'frame']


,Filename,tag,x1,y1,x2,y2,Origin file,Origin frame number,Origin track,Origin track frame number,clip,split,isNight,frame
0,dayTest/daySequence1--00000.jpg,stop,706,478,718,500,dayTest/daySequence1/Day1EveningShutter0.00020...,0,dayTest/daySequence1/Day1EveningShutter0.00020...,0,daySequence1,test,False,daySequence1--00000.jpg
1,dayTest/daySequence1--00001.jpg,stop,705,475,720,497,dayTest/daySequence1/Day1EveningShutter0.00020...,1,dayTest/daySequence1/Day1EveningShutter0.00020...,1,daySequence1,test,False,daySequence1--00001.jpg
2,dayTest/daySequence1--00002.jpg,stop,707,476,719,494,dayTest/daySequence1/Day1EveningShutter0.00020...,2,dayTest/daySequence1/Day1EveningShutter0.00020...,2,daySequence1,test,False,daySequence1--00002.jpg
3,dayTest/daySequence1--00005.jpg,stop,708,474,720,492,dayTest/daySequence1/Day1EveningShutter0.00020...,5,dayTest/daySequence1/Day1EveningShutter0.00020...,5,daySequence1,test,False,daySequence1--00005.jpg
4,dayTest/daySequence1--00006.jpg,stop,707,470,722,492,dayTest/daySequence1/Day1EveningShutter0.00020...,6,dayTest/daySequence1/Day1EveningShutter0.00020...,6,daySequence1,test,False,daySequence1--00006.jpg


## 2. Integrity checks

In [ ]:
ann["w"] = ann.x2 - ann.x1
ann["h"] = ann.y2 - ann.y1
print("Invalid boxes (w<=0 or h<=0):", ((ann.w <= 0) | (ann.h <= 0)).sum())
print("Boxes touching the frame border:", ((ann.x1 <= 0) | (ann.y1 <= 0)).sum())
print("Exact duplicate rows:", ann.duplicated(["frame", "tag", "x1", "y1", "x2", "y2"]).sum())
print("Unknown tags:", sorted(set(ann.tag) - set(CLASSES)))

> **Decision:** drop invalid/duplicate boxes; decide what to do with tags outside the six classes (e.g. `goForward`).

## 3. Class distribution – per clip, day/night, train/test

In [ ]:
tab = ann.pivot_table(index=["split", "clip"], columns="tag", values="frame",
                      aggfunc="size", fill_value=0)
tab["total"] = tab.sum(axis=1)
display(tab)

summary = ann.groupby(["split", "isNight", "tag"]).size().unstack(fill_value=0)
display(summary)
display((summary.T / summary.sum(axis=1)).T.round(3))   # class proportions

In [ ]:
# In how many TRAINING clips does each class appear?  Matters for grouped CV:
# a class present in only 2 clips cannot appear in every fold.
tr = ann[ann.split == "train"]
print(tr.groupby("tag")["clip"].nunique().sort_values())

> **Decision:**
> - Confirm totals quoted in the proposal (~51,800 training boxes, warning ≈ 2.5%).
> - Rare classes (likely `warningLeft`): note how many clips contain them → choose number of CV folds so each fold's validation set has every class, or report that class as "insufficient validation data".
> - Compare train vs test class proportions: a big shift should be mentioned in the report.

## 4. Box size and shape

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for night, g in ann.groupby("isNight"):
    lab = "night" if night else "day"
    ax[0].hist(g.h, bins=60, range=(0, 120), alpha=.6, label=lab)
    ax[1].hist(g.w, bins=60, range=(0, 60),  alpha=.6, label=lab)
    ax[2].hist(g.h / g.w, bins=60, range=(0, 5), alpha=.6, label=lab)
for a, t in zip(ax, ["height (px)", "width (px)", "aspect h/w"]):
    a.set_title(t); a.legend()
plt.tight_layout(); plt.show()
print(ann.groupby("tag")[["w", "h"]].describe(percentiles=[.05, .25, .5, .75]).round(1))

> **Decision:**
> - **Patch size**: choose a fixed size close to the median box (e.g. 16×32 if median ≈ 12×25) – upsampling tiny boxes far beyond their native size only adds blur.
> - **Minimum height filter**: pick a cutoff (e.g. 5th percentile) and record how many boxes it removes.
> - Lights with h/w far from ~2–3 may be horizontal lights or label errors → inspect in Section 6.

## 5. Temporal redundancy (how many *independent* samples do we really have?)

In [ ]:
fr = ann.groupby("clip")["frame"].nunique().rename("frames")
bx = ann.groupby("clip").size().rename("boxes")
print(pd.concat([fr, bx], axis=1))

# If the CSV contains track IDs, count physical lights (tracks) vs boxes
track_col = next((c for c in ann.columns if c.lower() == "origin track"), None)
if track_col:
    tracks = ann.groupby(["clip", track_col]).size()
    print(f"\nBoxes: {len(ann)}, tracks: {len(tracks)}, median boxes per track: {tracks.median():.0f}")
else:
    print("No track column found – estimate redundancy visually in Section 6.")

> **Decision:** frame subsampling step *k* (e.g. keep every 5th–10th frame). If a light appears in ~50 consecutive frames,
> the effective sample size is closer to the number of tracks than to the number of boxes – mention this in the report.

## 6. Visual inspection of crops

In [ ]:
# Index every frame once by basename -> full path (avoids guessing folder layouts)
FRAME_PATH = {os.path.basename(p): p for p in glob.glob(f"{ROOT}/**/frames/*.jpg", recursive=True)}
print(len(FRAME_PATH), "frames indexed")

def crop(r, margin=0.15):
    img = cv2.imread(FRAME_PATH[r.frame])
    mx, my = int(r.w * margin), int(r.h * margin)
    H, W = img.shape[:2]
    c = img[max(r.y1 - my, 0):min(r.y2 + my, H), max(r.x1 - mx, 0):min(r.x2 + mx, W)]
    return cv2.cvtColor(c, cv2.COLOR_BGR2RGB)

def grid(df, n=10, title=""):
    tags = [t for t in CLASSES if t in df.tag.unique()]
    fig, ax = plt.subplots(len(tags), n, figsize=(n * 1.1, len(tags) * 1.8), squeeze=False)
    for i, t in enumerate(tags):
        s = df[df.tag == t].sample(min(n, (df.tag == t).sum()), random_state=0)
        for j in range(n):
            a = ax[i, j]; a.axis("off")
            if j < len(s): a.imshow(crop(s.iloc[j]))
        ax[i, 0].set_title(t, loc="left", fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.show()

grid(ann[~ann.isNight], title="Day")
grid(ann[ann.isNight],  title="Night")

In [ ]:
# Smallest boxes: can a human still tell an arrow from a circle?
small = ann[ann.h <= ann.h.quantile(.10)]
grid(small, title="Smallest 10% of boxes")

> **Decision / notes to record:**
> - Label noise you can see (wrong class, box misaligned) → estimate a rough error rate from ~50 random crops.
> - At what box height do arrows become unreadable? That height is a natural break point for the "performance vs size" analysis.
> - Night artifacts (blooming, halos, reflections) → motivation for night-specific analysis and GMM class-conditionals.

## 7. Quick colour-feature preview (HSV)

In [ ]:
def lit_hue(rgb, s_min=80, v_min=120):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    m = (hsv[..., 1] >= s_min) & (hsv[..., 2] >= v_min)     # keep only bright, saturated (lit) pixels
    return hsv[..., 0][m]                                    # OpenCV hue range: 0–179

fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for a, night in zip(ax, [False, True]):
    for t in ["go", "stop", "warning"]:
        g = ann[(ann.isNight == night) & (ann.tag == t)]
        s = g.sample(min(300, len(g)), random_state=0)
        hues = np.concatenate([lit_hue(crop(r)) for r in s.itertuples()])
        a.hist(hues, bins=90, range=(0, 180), density=True, alpha=.5, label=t)
    a.set_title("Night" if night else "Day"); a.set_xlabel("hue"); a.legend()
plt.tight_layout(); plt.show()

> **Decision:**
> - **Red hue wraps around** (values near 0 *and* near 179). A plain hue histogram splits red into two distant bins; consider
>   a circular encoding (cos/sin of hue) or merging the two ends.
> - If day and night peaks of the same class differ, class-conditional densities are multimodal → supports GMM.
> - If the three colours are already perfectly separated, colour alone saturates the 3-class task → confirms the six-class / shape focus.

## 8. Summary of decisions (fill in)
| Item | Value | Evidence |
|---|---|---|
| Classes kept / handling of `goForward` | | Sec. 2–3 |
| Number of CV folds, fold assignment | | Sec. 3 |
| Patch size | | Sec. 4 |
| Minimum box height (boxes removed) | | Sec. 4 |
| Frame subsampling step *k* | | Sec. 5 |
| Estimated label-noise rate | | Sec. 6 |
| Hue encoding | | Sec. 7 |